# Imports

In [104]:
import numpy as np
import pandas as pd
from scipy.io import loadmat
import pywt

from scipy.stats import skew
from scipy.stats import kurtosis

from scipy.fft import fft
from scipy.fft import fftfreq

# Sampling Constants

In [105]:
SAMPLING_RATE = 12000

WINDOW_SIZE = 1024

STEP_SIZE = 512

# Fault Frequencies

In [106]:
BPFO = 107

BPFI = 162

BSF = 141

# File Mapping

In [107]:
file_mapping = {

    # NORMAL

    97:0,
    98:0,
    99:0,
    100:0,

    # INNER RACE

    105:1,
    106:1,
    107:1,
    108:1,

    # BALL

    118:2,
    119:2,
    120:2,
    121:2,

    # OUTER

    130:3,
    131:3,
    132:3,
    133:3

}

# Load Signal Function

In [108]:
def load_cwru_signal(filename, variable_name):

    data = loadmat(filename)

    signal = data[variable_name]

    signal = signal.flatten()

    return signal

# RPM Function

In [109]:
def get_rpm(filename):

    data = loadmat(filename)

    rpm_keys = [
        k for k in data.keys()
        if "RPM" in k.upper()
    ]

    if len(rpm_keys) == 0:

        return None

    return data[rpm_keys[0]].item()

# Windowing Function

In [110]:
def create_windows(signal):

    windows = []

    for start in range(
        0,
        len(signal)-WINDOW_SIZE,
        STEP_SIZE
    ):

        window = signal[
            start:start+WINDOW_SIZE
        ]

        windows.append(window)

    return windows

# FFT Helper

In [111]:
def get_fft(window):

    N = len(window)

    fft_values = np.abs(
        fft(window)
    )[:N//2]

    freqs = fftfreq(
        N,
        d=1/SAMPLING_RATE
    )[:N//2]

    return freqs, fft_values

# Frequency Helpers

In [112]:
def amplitude_at_frequency(
    freqs,
    spectrum,
    target_freq
):

    idx = np.argmin(
        np.abs(freqs-target_freq)
    )

    return spectrum[idx]
# Band Power

def band_power(
    freqs,
    spectrum,
    low,
    high
):

    mask = (
        (freqs>=low)
        &
        (freqs<high)
    )

    return np.sum(
        spectrum[mask]
    )

# Wavelet Energies

In [113]:
def wavelet_energies(window):

    coeffs = pywt.wavedec(
        window,
        'db4',
        level=4
    )

    cA4,d4,d3,d2,d1 = coeffs

    return [

        np.sum(d1**2),

        np.sum(d2**2),

        np.sum(d3**2),

        np.sum(d4**2)

    ]

# Time Domain Feature

In [114]:
def time_features(window):

    mean = np.mean(window)

    rms = np.sqrt(np.mean(window**2))

    std = np.std(window)

    peak = np.max(np.abs(window))

    peak_to_peak = np.ptp(window)

    skewness = skew(window)

    kurt = kurtosis(window)

    crest_factor = peak / rms

    shape_factor = rms / np.mean(np.abs(window))

    impulse_factor = peak / np.mean(np.abs(window))

    return {

        "Mean": mean,
        "RMS": rms,
        "Std": std,
        "Peak": peak,
        "Peak_to_Peak": peak_to_peak,
        "Skewness": skewness,
        "Kurtosis": kurt,
        "Crest_Factor": crest_factor,
        "Shape_Factor": shape_factor,
        "Impulse_Factor": impulse_factor

    }

In [115]:
print(time_features)

<function time_features at 0x0000025B7D231B20>


# Master Feature Extraction

In [116]:
def extract_features(window):

    time_feats = time_features(window)

    freqs, spectrum = get_fft(window)

    spectral_energy = np.sum(
        spectrum**2
    )

    dominant_frequency = freqs[
        np.argmax(spectrum)
    ]

    amp_bpfo = amplitude_at_frequency(
        freqs,
        spectrum,
        BPFO
    )

    amp_bpfi = amplitude_at_frequency(
        freqs,
        spectrum,
        BPFI
    )

    amp_bsf = amplitude_at_frequency(
        freqs,
        spectrum,
        BSF
    )

    bpfo_bpfi_ratio = (
        amp_bpfo / amp_bpfi
        if amp_bpfi != 0
        else 0
    )

    band_0_500 = band_power(
        freqs,
        spectrum,
        0,
        500
    )

    band_500_1000 = band_power(
        freqs,
        spectrum,
        500,
        1000
    )

    band_1000_2000 = band_power(
        freqs,
        spectrum,
        1000,
        2000
    )

    e_d1, e_d2, e_d3, e_d4 = wavelet_energies(window)

    return {

        **time_feats,

        "Spectral_Energy": spectral_energy,

        "Dominant_Frequency": dominant_frequency,

        "Amp_BPFO": amp_bpfo,

        "Amp_BPFI": amp_bpfi,

        "Amp_BSF": amp_bsf,

        "BPFO_BPFI_Ratio": bpfo_bpfi_ratio,

        "Band_0_500": band_0_500,

        "Band_500_1000": band_500_1000,

        "Band_1000_2000": band_1000_2000,

        "Wavelet_D1": e_d1,

        "Wavelet_D2": e_d2,

        "Wavelet_D3": e_d3,

        "Wavelet_D4": e_d4
    }

# Dataset Builder

In [117]:
all_rows = []

# Main Processing Loop

In [118]:
for file_id,label in file_mapping.items():

    filename = f"{file_id}.mat"

    print(
        f"Processing {filename}"
    )

    data = loadmat(filename)

    de_key = [
        k for k in data.keys()
        if "DE_time" in k
    ][0]

    signal = (
        data[de_key]
        .flatten()
    )

    rpm = get_rpm(filename)

    windows = create_windows(signal)

    for window in windows:

        features = extract_features(
            window
        )

        features["Label"] = label

        features["RPM"] = rpm

        features["Source_File"] = filename

        all_rows.append(
            features
        )

Processing 97.mat
Processing 98.mat
Processing 99.mat
Processing 100.mat
Processing 105.mat
Processing 106.mat
Processing 107.mat
Processing 108.mat
Processing 118.mat
Processing 119.mat
Processing 120.mat
Processing 121.mat
Processing 130.mat
Processing 131.mat
Processing 132.mat
Processing 133.mat


# Create DataFrame

In [119]:
df = pd.DataFrame(all_rows)

df.head()

,Mean,RMS,Std,Peak,Peak_to_Peak,Skewness,Kurtosis,Crest_Factor,Shape_Factor,Impulse_Factor,...,Band_0_500,Band_500_1000,Band_1000_2000,Wavelet_D1,Wavelet_D2,Wavelet_D3,Wavelet_D4,Label,RPM,Source_File
0,0.012421,0.075982,0.074960,0.217794,0.416396,-0.162976,-0.243664,2.866393,1.232692,3.533380,...,147.551131,95.658846,161.241849,0.036103,0.559792,3.431540,0.423184,0,1796.0,97.mat
1,0.012693,0.073679,0.072577,0.272869,0.485031,-0.156544,0.021868,3.703495,1.247336,4.619501,...,146.979278,79.725276,165.712824,0.037583,0.547612,3.135956,0.373435,0,1796.0,97.mat
2,0.011589,0.070425,0.069465,0.272869,0.485031,-0.107386,0.184817,3.874580,1.262097,4.890095,...,143.471074,90.892121,139.916615,0.038085,0.537930,2.639634,0.416981,0,1796.0,97.mat
3,0.011424,0.072956,0.072056,0.211536,0.386147,0.081507,-0.304845,2.899512,1.251572,3.628947,...,140.360081,87.465467,163.903410,0.039168,0.596414,3.155051,0.418865,0,1796.0,97.mat
4,0.014643,0.078377,0.076996,0.212370,0.420569,0.050329,-0.376182,2.709618,1.242484,3.366659,...,152.797162,64.809815,170.830579,0.038150,0.632442,3.519659,0.337282,0,1796.0,97.mat


# Check Dataset Shape

In [120]:
print(df.shape)

(6153, 26)


In [121]:
df["Label"].value_counts()

Label
0    3310
3     949
1     948
2     946
Name: count, dtype: int64

# Dataset Distribution by Source File and Fault Class

In [122]:
df.groupby(
    ["Source_File","Label"]
).size()

Source_File  Label
100.mat      0        947
105.mat      1        235
106.mat      1        237
107.mat      1        237
108.mat      1        239
118.mat      2        238
119.mat      2        236
120.mat      2        236
121.mat      2        236
130.mat      3        237
131.mat      3        238
132.mat      3        236
133.mat      3        238
97.mat       0        475
98.mat       0        944
99.mat       0        944
dtype: int64

# Missing Value Analysis of the Extracted Feature Dataset

In [123]:
df.isnull().sum()

Mean                     0
RMS                      0
Std                      0
Peak                     0
Peak_to_Peak             0
Skewness                 0
Kurtosis                 0
Crest_Factor             0
Shape_Factor             0
Impulse_Factor           0
Spectral_Energy          0
Dominant_Frequency       0
Amp_BPFO                 0
Amp_BPFI                 0
Amp_BSF                  0
BPFO_BPFI_Ratio          0
Band_0_500               0
Band_500_1000            0
Band_1000_2000           0
Wavelet_D1               0
Wavelet_D2               0
Wavelet_D3               0
Wavelet_D4               0
Label                    0
RPM                   1888
Source_File              0
dtype: int64

# Drop RPM

In [124]:
df = df.drop(columns=["RPM"])

# Save Data Set

In [125]:
df.to_csv(
    "phase1_full_dataset.csv",
    index=False
)

print("Dataset saved successfully")

Dataset saved successfully


In [126]:
df.head()
df.shape
df.info()
df["Label"].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6153 entries, 0 to 6152
Data columns (total 25 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Mean                6153 non-null   float64
 1   RMS                 6153 non-null   float64
 2   Std                 6153 non-null   float64
 3   Peak                6153 non-null   float64
 4   Peak_to_Peak        6153 non-null   float64
 5   Skewness            6153 non-null   float64
 6   Kurtosis            6153 non-null   float64
 7   Crest_Factor        6153 non-null   float64
 8   Shape_Factor        6153 non-null   float64
 9   Impulse_Factor      6153 non-null   float64
 10  Spectral_Energy     6153 non-null   float64
 11  Dominant_Frequency  6153 non-null   float64
 12  Amp_BPFO            6153 non-null   float64
 13  Amp_BPFI            6153 non-null   float64
 14  Amp_BSF             6153 non-null   float64
 15  BPFO_BPFI_Ratio     6153 non-null   float64
 16  Band_0

Label
0    3310
3     949
1     948
2     946
Name: count, dtype: int64